# Features Selection — Van Zieleghem, Dai Pra

## Table des matières
1. [Imports et chargement des données](#imports)
2. [Sélection Personnelle (P)](#perso)
3. [SelectKBest (KB)](#kbest)
4. [Variance Threshold (VT)](#vt)
5. [Résumé comparatif](#resume)

## 1. Imports et chargement des données <a id='imports'></a>

On charge les 4 fichiers CSV produits par le preprocessing (X_train, X_test, y_train, y_test) et les librairies de sélection de features.

In [1]:
# librairies de manipulation de données
import pandas as pd
import numpy as np

# méthodes de sélection de features
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.feature_selection import VarianceThreshold

# fonction utilitaire
def list_to_string(liste):
    """Convertit une liste en chaîne séparée par des virgules."""
    return ",".join(str(e) for e in liste)

# chargement des données préprocessées
df_X_train = pd.read_csv('X_train.csv', header=0)
df_X_test = pd.read_csv('X_test.csv', header=0)

df_y_train = pd.read_csv('y_train.csv', header=0)
df_y_test = pd.read_csv('y_test.csv', header=0)

# conversion en numpy
y_train = df_y_train.to_numpy().ravel()
y_test = df_y_test.to_numpy().ravel()

print("X_train :", df_X_train.shape)
print("X_test  :", df_X_test.shape)
print("\nColonnes disponibles :", list(df_X_train.columns))

X_train : (1047, 18)
X_test  : (262, 18)

Colonnes disponibles : ['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'HasCabin']


Les données préprocessées sont chargées. On dispose de toutes les features encodées et standardisées. On va maintenant appliquer 3 méthodes de sélection de features pour réduire la dimensionnalité et améliorer les performances des modèles.

## 2. Sélection Personnelle (P) <a id='perso'></a>

On sélectionne manuellement les variables les plus pertinentes d'après l'analyse exploratoire (Phase 1) :
- **sex** (female/male) : variable la plus discriminante (∸73% survie femmes vs ∸19% hommes)
- **pclass** : forte corrélation négative avec la survie (1ère classe = meilleure survie)
- **fare** : lié à la classe sociale, corrélé positivement avec la survie
- **HasCabin** : proxy du statut social (cabine renseignée = passager plus aisé)
- **IsAlone** : voyager seul diminue les chances de survie

On choisit ces variables car elles ont montré les corrélations les plus fortes avec `survived` dans la matrice de corrélation et les analyses bivariées de Phase 1.

In [2]:
# sélection des colonnes pertinentes (incluant les colonnes OneHot de sex)
perso_features = [col for col in df_X_train.columns if (
    'sex' in col or
    'pclass' in col or
    'fare' in col or
    'HasCabin' in col or
    'IsAlone' in col
)]

print("Features sélectionnées (P) :", perso_features)
print("Nombre de features :", len(perso_features))

# extraction des sous-ensembles
P_X_train = df_X_train[perso_features].to_numpy()
P_X_test = df_X_test[perso_features].to_numpy()

# sauvegarde
header = list_to_string(perso_features)
np.savetxt("xtrain_P.csv", P_X_train, delimiter=",", fmt="%.4f", header=header, comments="")
np.savetxt("xtest_P.csv", P_X_test, delimiter=",", fmt="%.4f", header=header, comments="")

print("\nFichiers sauvegardés : xtrain_P.csv, xtest_P.csv")
print("Shape train :", P_X_train.shape)
print("Shape test  :", P_X_test.shape)

Features sélectionnées (P) : ['sex_female', 'sex_male', 'pclass', 'fare', 'IsAlone', 'HasCabin']
Nombre de features : 6

Fichiers sauvegardés : xtrain_P.csv, xtest_P.csv
Shape train : (1047, 6)
Shape test  : (262, 6)


La sélection personnelle retient les variables identifiées comme les plus importantes lors de l'analyse exploratoire. Cette approche repose sur la connaissance du domaine et l'interprétation des résultats statistiques de la Phase 1.

## 3. SelectKBest (KB) <a id='kbest'></a>

On utilise la méthode `SelectKBest` avec le test statistique ANOVA (`f_classif`) pour sélectionner les **k=6** meilleures features.

Le choix de k=6 est motivé par le fait qu'on souhaite réduire significativement la dimensionnalité (environ la moitié des features) tout en conservant suffisamment d'information. ANOVA mesure la séparation entre les classes pour chaque feature individuellement.

In [3]:
# sélection des 6 meilleures features par ANOVA
selector_kb = SelectKBest(f_classif, k=6)

KB_X_train = selector_kb.fit_transform(df_X_train, y_train)
KB_X_test = selector_kb.transform(df_X_test)

# récupération des features sélectionnées
selected_mask_kb = selector_kb.get_support()
selected_features_kb = df_X_train.columns[selected_mask_kb].tolist()

# affichage des scores ANOVA pour toutes les features
scores_df = pd.DataFrame({
    'Feature': df_X_train.columns,
    'F-score': selector_kb.scores_,
    'p-value': selector_kb.pvalues_,
    'Sélectionnée': selected_mask_kb
}).sort_values('F-score', ascending=False)

print("Scores ANOVA (f_classif) pour chaque feature :")
print(scores_df.to_string(index=False))

print(f"\nFeatures sélectionnées (KB, k=6) : {selected_features_kb}")

# sauvegarde
header = list_to_string(selected_features_kb)
np.savetxt("xtrain_KB.csv", KB_X_train, delimiter=",", fmt="%.4f", header=header, comments="")
np.savetxt("xtest_KB.csv", KB_X_test, delimiter=",", fmt="%.4f", header=header, comments="")

print("\nFichiers sauvegardés : xtrain_KB.csv, xtest_KB.csv")
print("Shape train :", KB_X_train.shape)
print("Shape test  :", KB_X_test.shape)

Scores ANOVA (f_classif) pour chaque feature :
     Feature    F-score      p-value  Sélectionnée
  sex_female 411.343343 2.232027e-77          True
    sex_male 411.343343 2.232027e-77          True
    Title_Mr 397.851476 2.923524e-75          True
   Title_Mrs 149.142999 3.693781e-32          True
      pclass 119.019583 2.557951e-26          True
  Title_Miss 106.028259 9.483044e-24          True
    HasCabin 100.763988 1.064165e-22         False
        fare  68.564197 3.724075e-16         False
  embarked_C  35.681494 3.182093e-09         False
     IsAlone  31.640048 2.381468e-08         False
  embarked_S  24.127825 1.045697e-06         False
       parch   4.787792 2.888300e-02         False
       sibsp   3.405076 6.527889e-02         False
Title_Master   1.542398 2.145399e-01         False
  embarked_Q   0.203533 6.519773e-01         False
         age   0.091346 7.625336e-01         False
  Title_Rare   0.034912 8.518170e-01         False
  FamilySize   0.011157 9.158986e-0

Le test ANOVA classe les features par leur capacité à séparer les deux classes (survie/non-survie). Les p-values faibles (< 0.05) indiquent des features statistiquement significatives. Les 6 features retenues sont celles avec les F-scores les plus élevés. On remarquera que certaines features sélectionnées peuvent différer de la sélection personnelle — c'est normal car ANOVA ne prend pas en compte les corrélations entre features.

## 4. Variance Threshold (VT) <a id='vt'></a>

On utilise `VarianceThreshold` pour éliminer les features dont la variance est inférieure à un seuil.

Après standardisation, les features ont en principe une variance proche de 1. On choisit un seuil très bas (`threshold=0.01`) pour éliminer uniquement les features quasi-constantes (variance négligeable). Ce seuil conserve la majorité des features — seules celles qui n'apportent aucune information discriminante seront retirées.

In [4]:
# VT s'applique sur les données BRUTES (non standardisées) pour détecter
# les features à faible variance de manière significative.
# Mais on sauvegarde la version STANDARDISÉE des features sélectionnées
# pour rester compatible avec KNN dans les notebooks de modélisation.

# 1. Chargement des données brutes (uniquement pour calculer les variances)
df_X_train_brut = pd.read_csv('X_train_brut.csv', header=0)

# 2. Application de VT sur les données brutes pour identifier les features à garder
selector_vt = VarianceThreshold(threshold=0.01)
selector_vt.fit(df_X_train_brut)

selected_mask_vt = selector_vt.get_support()
selected_features_vt = df_X_train_brut.columns[selected_mask_vt].tolist()

# 3. Affichage des variances brutes pour transparence
variances_df = pd.DataFrame({
    'Feature'     : df_X_train_brut.columns,
    'Variance'    : selector_vt.variances_,
    'Sélectionnée': selected_mask_vt
}).sort_values('Variance', ascending=False)

print("Variance de chaque feature (calculée sur données brutes) :")
print(variances_df.to_string(index=False))
print(f"Features sélectionnées (VT, threshold=0.01) : {selected_features_vt}")
print(f"Features éliminées : {df_X_train_brut.columns[~selected_mask_vt].tolist()}")

# 4. Extraction de la version STANDARDISÉE des features sélectionnées
#    (df_X_train et df_X_test sont chargés depuis X_train.csv / X_test.csv au début du notebook)
VT_X_train = df_X_train[selected_features_vt].to_numpy()
VT_X_test  = df_X_test[selected_features_vt].to_numpy()

# 5. Sauvegarde
header = list_to_string(selected_features_vt)
np.savetxt("xtrain_VT.csv", VT_X_train, delimiter=",", fmt="%.4f", header=header, comments="")
np.savetxt("xtest_VT.csv",  VT_X_test,  delimiter=",", fmt="%.4f", header=header, comments="")

print("Fichiers sauvegardés : xtrain_VT.csv, xtest_VT.csv")
print("Shape train :", VT_X_train.shape)
print("Shape test  :", VT_X_test.shape)

Variance de chaque feature (calculée sur données brutes) :
     Feature    Variance  Sélectionnée
        fare 2612.620382          True
         age  157.678173          True
  FamilySize    2.644107          True
       sibsp    1.198786          True
       parch    0.714309          True
      pclass    0.707374          True
    Title_Mr    0.245336          True
     IsAlone    0.240602          True
    sex_male    0.232248          True
  sex_female    0.232248          True
  embarked_S    0.205244          True
    HasCabin    0.178736          True
  Title_Miss    0.163742          True
  embarked_C    0.157461          True
   Title_Mrs    0.132103          True
  embarked_Q    0.084062          True
Title_Master    0.042875          True
  Title_Rare    0.023308          True
Features sélectionnées (VT, threshold=0.01) : ['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'pclass', 'age',

Avec un seuil de 0.01, `VarianceThreshold` élimine les features dont la **variance brute** (avant standardisation) est quasi-nulle. On calcule les variances sur les données brutes pour que le filtre soit significatif — après standardisation, toutes les features numériques auraient une variance ≈ 1, ce qui rendrait VT inopérant. On sauvegarde ensuite la **version standardisée** des features sélectionnées pour rester compatible avec les algorithmes sensibles à l'échelle comme KNN. Cette méthode est non-supervisée (elle n'utilise pas la cible `y`), contrairement à `SelectKBest`.

## 5. Résumé comparatif <a id='resume'></a>

Comparaison des 3 méthodes de sélection de features.

In [5]:
# résumé comparatif
print("=" * 60)
print("RÉSUMÉ DES 3 MÉTHODES DE SÉLECTION")
print("=" * 60)

print(f"\n1. Sélection Personnelle (P) : {len(perso_features)} features")
print(f"   → {perso_features}")

print(f"\n2. SelectKBest (KB, k=6) : {len(selected_features_kb)} features")
print(f"   → {selected_features_kb}")

print(f"\n3. VarianceThreshold (VT, t=0.01) : {len(selected_features_vt)} features")
print(f"   → {selected_features_vt}")

# features communes aux 3 méthodes
common = set(perso_features) & set(selected_features_kb) & set(selected_features_vt)
print(f"\nFeatures communes aux 3 méthodes : {common}")

print("\n" + "=" * 60)
print("Fichiers CSV générés : 6 fichiers")
print("  - xtrain_P.csv, xtest_P.csv")
print("  - xtrain_KB.csv, xtest_KB.csv")
print("  - xtrain_VT.csv, xtest_VT.csv")
print("=" * 60)

RÉSUMÉ DES 3 MÉTHODES DE SÉLECTION

1. Sélection Personnelle (P) : 6 features
   → ['sex_female', 'sex_male', 'pclass', 'fare', 'IsAlone', 'HasCabin']

2. SelectKBest (KB, k=6) : 6 features
   → ['sex_female', 'sex_male', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'pclass']

3. VarianceThreshold (VT, t=0.01) : 18 features
   → ['sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'pclass', 'age', 'sibsp', 'parch', 'fare', 'FamilySize', 'IsAlone', 'HasCabin']

Features communes aux 3 méthodes : {'sex_female', 'sex_male', 'pclass'}

Fichiers CSV générés : 6 fichiers
  - xtrain_P.csv, xtest_P.csv
  - xtrain_KB.csv, xtest_KB.csv
  - xtrain_VT.csv, xtest_VT.csv


Chaque méthode de sélection a ses forces :
- **Personnelle (P)** : basée sur la connaissance du domaine et l'analyse exploratoire — interprétable et justifiée
- **SelectKBest (KB)** : sélection statistique automatique — objective mais ne tient pas compte des corrélations inter-features
- **VarianceThreshold (VT)** : filtre les features non-informatives — simple et rapide mais ne mesure pas la relation avec la cible

Les 6 fichiers CSV générés seront utilisés dans l'étape de **modélisation** avec les 3 algorithmes (KNN, Decision Tree, Random Forest).